# DQN Experiments - ALE/Boxing-v5

**Member:** Nice Eva Karabaranga  
**Environment:** ALE/Boxing-v5  
**Policy:** CnnPolicy (with MlpPolicy ablation in Exp 10)  
**Total Experiments:** 10  

## 1. Install & Import Dependencies

In [5]:
# Install dependencies
!pip install stable-baselines3[extra] gymnasium[atari] ale-py -q
!pip install "autorom[accept-rom-license]" -q

In [8]:
import os, csv, json, time, shutil
from pathlib import Path

import ale_py
import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecMonitor

## 2. Experiment Configurations

10 distinct hyperparameter sets — deliberately varied from Elyse's experiments.  
Some use `epsilon_end=0` or `gamma=0` to probe extreme behavior.

In [9]:
EXPERIMENTS = [
    {
        "name": "exp01_baseline",
        "label": "Exp 1 – Baseline",
        "policy": "CnnPolicy",
        "lr": 2.5e-4,
        "gamma": 0.95,
        "batch_size": 64,
        "eps_start": 1.0,
        "eps_end": 0.02,
        "eps_fraction": 0.20,
        "note": "Moderate LR, mid gamma, no extreme settings — solid reference point."
    },
    {
        "name": "exp02_zero_eps_end",
        "label": "Exp 2 – Zero Epsilon End",
        "policy": "CnnPolicy",
        "lr": 2.5e-4,
        "gamma": 0.95,
        "batch_size": 64,
        "eps_start": 1.0,
        "eps_end": 0.0,
        "eps_fraction": 0.20,
        "note": "Fully greedy at end — zero residual exploration. Tests pure exploitation."
    },
    {
        "name": "exp03_zero_gamma",
        "label": "Exp 3 – Zero Gamma",
        "policy": "CnnPolicy",
        "lr": 2.5e-4,
        "gamma": 0.0,
        "batch_size": 64,
        "eps_start": 1.0,
        "eps_end": 0.02,
        "eps_fraction": 0.20,
        "note": "Agent is fully myopic — ignores all future rewards. Expected very poor performance."
    },
    {
        "name": "exp04_very_high_lr",
        "label": "Exp 4 – Very High LR",
        "policy": "CnnPolicy",
        "lr": 1e-3,
        "gamma": 0.95,
        "batch_size": 64,
        "eps_start": 1.0,
        "eps_end": 0.02,
        "eps_fraction": 0.20,
        "note": "Aggressive LR — likely unstable Q-values, oscillating rewards."
    },
    {
        "name": "exp05_tiny_lr",
        "label": "Exp 5 – Tiny LR",
        "policy": "CnnPolicy",
        "lr": 1e-6,
        "gamma": 0.95,
        "batch_size": 64,
        "eps_start": 1.0,
        "eps_end": 0.02,
        "eps_fraction": 0.20,
        "note": "Near-zero LR — agent barely learns; stable but stagnant."
    },
    {
        "name": "exp06_large_batch_high_gamma",
        "label": "Exp 6 – Large Batch + High Gamma",
        "policy": "CnnPolicy",
        "lr": 2.5e-4,
        "gamma": 0.999,
        "batch_size": 256,
        "eps_start": 1.0,
        "eps_end": 0.02,
        "eps_fraction": 0.20,
        "note": "Stable gradient estimates combined with strong future valuation — may favour long-term planning."
    },
    {
        "name": "exp07_instant_exploit",
        "label": "Exp 7 – Instant Exploitation",
        "policy": "CnnPolicy",
        "lr": 2.5e-4,
        "gamma": 0.95,
        "batch_size": 64,
        "eps_start": 1.0,
        "eps_end": 0.02,
        "eps_fraction": 0.01,
        "note": "Epsilon collapses in the first 1% of training — almost no exploration phase."
    },
    {
        "name": "exp08_full_explore",
        "label": "Exp 8 – Full Exploration",
        "policy": "CnnPolicy",
        "lr": 2.5e-4,
        "gamma": 0.95,
        "batch_size": 64,
        "eps_start": 1.0,
        "eps_end": 0.02,
        "eps_fraction": 0.90,
        "note": "Exploration lasts 90% of training — agent learns slowly but broadly."
    },
    {
        "name": "exp09_best_guess",
        "label": "Exp 9 – Best Config",
        "policy": "CnnPolicy",
        "lr": 2.5e-4,
        "gamma": 0.99,
        "batch_size": 64,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_fraction": 0.15,
        "note": "Balanced config tuned from prior experiments — expected highest reward."
    },
    {
        "name": "exp10_mlp_ablation",
        "label": "Exp 10 – MLP Ablation",
        "policy": "MlpPolicy",
        "lr": 2.5e-4,
        "gamma": 0.99,
        "batch_size": 64,
        "eps_start": 1.0,
        "eps_end": 0.01,
        "eps_fraction": 0.15,
        "note": "Same config as Exp 9 but MlpPolicy on RAM obs — tests CNN vs MLP architecture."
    },
]

print(f" {len(EXPERIMENTS)} experiments configured.")
for e in EXPERIMENTS:
    print(f"  {e['name']:35s} | policy={e['policy']:10s} | lr={e['lr']:.2e} | gamma={e['gamma']} | batch={e['batch_size']} | eps=[{e['eps_start']},{e['eps_end']},{e['eps_fraction']}]")

 10 experiments configured.
  exp01_baseline                      | policy=CnnPolicy  | lr=2.50e-04 | gamma=0.95 | batch=64 | eps=[1.0,0.02,0.2]
  exp02_zero_eps_end                  | policy=CnnPolicy  | lr=2.50e-04 | gamma=0.95 | batch=64 | eps=[1.0,0.0,0.2]
  exp03_zero_gamma                    | policy=CnnPolicy  | lr=2.50e-04 | gamma=0.0 | batch=64 | eps=[1.0,0.02,0.2]
  exp04_very_high_lr                  | policy=CnnPolicy  | lr=1.00e-03 | gamma=0.95 | batch=64 | eps=[1.0,0.02,0.2]
  exp05_tiny_lr                       | policy=CnnPolicy  | lr=1.00e-06 | gamma=0.95 | batch=64 | eps=[1.0,0.02,0.2]
  exp06_large_batch_high_gamma        | policy=CnnPolicy  | lr=2.50e-04 | gamma=0.999 | batch=256 | eps=[1.0,0.02,0.2]
  exp07_instant_exploit               | policy=CnnPolicy  | lr=2.50e-04 | gamma=0.95 | batch=64 | eps=[1.0,0.02,0.01]
  exp08_full_explore                  | policy=CnnPolicy  | lr=2.50e-04 | gamma=0.95 | batch=64 | eps=[1.0,0.02,0.9]
  exp09_best_guess                 

## 3. Helper Functions

In [10]:
ENV_ID     = "ALE/Boxing-v5"
RAM_ENV_ID = "ALE/Boxing-ram-v5"
SEED       = 42
TOTAL_TS   = 100_000
EVAL_EPS   = 5
EVAL_FREQ  = 10_000
BUFFER     = 50_000
LRN_STARTS = 2_000

OUT_DIR = Path("results/Nice")
OUT_DIR.mkdir(parents=True, exist_ok=True)


class EpisodeLogger(BaseCallback):
    """Collect episode rewards & lengths during training."""
    def __init__(self):
        super().__init__()
        self.records = []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.records.append({
                    "timestep": self.num_timesteps,
                    "reward":   info["episode"]["r"],
                    "length":   info["episode"]["l"],
                })
        return True


def make_cnn_env(seed=SEED, render_mode=None):
    env = make_atari_env(ENV_ID, n_envs=1, seed=seed,
                         env_kwargs={"render_mode": render_mode})
    env = VecMonitor(env)
    return VecFrameStack(env, n_stack=4)


def make_mlp_env(seed=SEED, render_mode=None):
    def _init():
        e = gym.make(RAM_ENV_ID, render_mode=render_mode)
        return Monitor(e)
    env = DummyVecEnv([_init])
    return VecMonitor(env)


def build_env(policy, seed, render_mode=None):
    return make_cnn_env(seed, render_mode) if policy == "CnnPolicy" \
           else make_mlp_env(seed, render_mode)


print("Helpers ready.")

Helpers ready.


## 4. Run All 10 Experiments

 Each experiment runs for `TOTAL_TS` timesteps.

In [11]:
results = []   # list of dicts — one per experiment
all_curves = {}  # exp_name -> list of (timestep, reward)

best_mean   = -np.inf
best_exp    = None

for cfg in EXPERIMENTS:
    name   = cfg["name"]
    policy = cfg["policy"]
    print(f"\n{'='*60}")
    print(f" {name}  |  policy={policy}")
    print(f"  lr={cfg['lr']:.2e}  gamma={cfg['gamma']}  batch={cfg['batch_size']}")
    print(f"  eps=[{cfg['eps_start']}, {cfg['eps_end']}, {cfg['eps_fraction']}]")
    print(f"{'='*60}")

    train_env = build_env(policy, SEED)
    eval_env  = build_env(policy, SEED + 100)
    final_env = build_env(policy, SEED + 200)

    ep_logger = EpisodeLogger()
    best_dir  = OUT_DIR / f"{name}_best"
    eval_cb   = EvalCallback(
        eval_env,
        best_model_save_path=str(best_dir),
        eval_freq=EVAL_FREQ,
        n_eval_episodes=EVAL_EPS,
        deterministic=True,
        render=False,
        verbose=0,
    )

    model = DQN(
        policy=policy,
        env=train_env,
        learning_rate=cfg["lr"],
        gamma=cfg["gamma"],
        batch_size=cfg["batch_size"],
        buffer_size=BUFFER,
        learning_starts=LRN_STARTS,
        train_freq=4,
        gradient_steps=1,
        target_update_interval=10_000,
        exploration_initial_eps=cfg["eps_start"],
        exploration_final_eps=cfg["eps_end"],
        exploration_fraction=cfg["eps_fraction"],
        seed=SEED,
        device="auto",
        verbose=0,
    )

    t0 = time.time()
    model.learn(
        total_timesteps=TOTAL_TS,
        callback=[ep_logger, eval_cb],
        progress_bar=True,
    )
    elapsed = (time.time() - t0) / 60

    # Save final model
    model_path = OUT_DIR / f"{name}.zip"
    model.save(str(model_path.with_suffix("")))

    # Final eval
    mean_r, std_r = evaluate_policy(model, final_env,
                                     n_eval_episodes=EVAL_EPS,
                                     deterministic=True)

    all_curves[name] = ep_logger.records

    row = {
        "experiment":    name,
        "label":         cfg["label"],
        "policy":        policy,
        "lr":            cfg["lr"],
        "gamma":         cfg["gamma"],
        "batch_size":    cfg["batch_size"],
        "eps_start":     cfg["eps_start"],
        "eps_end":       cfg["eps_end"],
        "eps_fraction":  cfg["eps_fraction"],
        "mean_reward":   round(float(mean_r), 2),
        "std_reward":    round(float(std_r),  2),
        "train_minutes": round(elapsed, 2),
        "note":          cfg["note"],
    }
    results.append(row)

    print(f" mean_reward={mean_r:.2f} ± {std_r:.2f}  |  {elapsed:.1f} min")

    # Track global best
    if mean_r > best_mean:
        best_mean = mean_r
        best_exp  = name
        best_model_src = str(model_path)

    train_env.close()
    eval_env.close()
    final_env.close()

print(f"\n Best experiment: {best_exp}  (mean_reward={best_mean:.2f})")


 exp01_baseline  |  policy=CnnPolicy
  lr=2.50e-04  gamma=0.95  batch=64
  eps=[1.0, 0.02, 0.2]


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Output()

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7e9692a1b380> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7e969372bd40>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


 mean_reward=-0.20 ± 7.36  |  13.2 min

 exp02_zero_eps_end  |  policy=CnnPolicy
  lr=2.50e-04  gamma=0.95  batch=64
  eps=[1.0, 0.0, 0.2]


...

 mean_reward=0.00 ± 6.07  |  12.4 min

 exp09_best_guess  |  policy=CnnPolicy
  lr=2.50e-04  gamma=0.99  batch=64
  eps=[1.0, 0.01, 0.15]


Output()

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7e95cef46d50> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7e95cf02f0b0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


 mean_reward=1.00 ± 2.61  |  13.1 min

 exp10_mlp_ablation  |  policy=MlpPolicy
  lr=2.50e-04  gamma=0.99  batch=64
  eps=[1.0, 0.01, 0.15]


NameNotFound: Environment `Boxing-ram` doesn't exist in namespace ALE. Did you mean: `Boxing`?

## 5. Save Best Model

In [ ]:
best_dest = "Nice_dqn_model.zip"
shutil.copy(best_model_src, best_dest)
print(f" Best model saved → {best_dest}")
print(f"   Source: {best_model_src}")
print(f"   Experiment: {best_exp}  |  mean_reward={best_mean:.2f}")

## 6. Save Hyperparameter Table

In [ ]:
df = pd.DataFrame(results)

csv_path = "hyperparameter_table_nice.csv"
df.to_csv(csv_path, index=False)
print(f" CSV saved → {csv_path}")

display_cols = ["experiment", "policy", "lr", "gamma", "batch_size",
                "eps_start", "eps_end", "eps_fraction", "mean_reward", "std_reward"]
df[display_cols].style.highlight_max(subset=["mean_reward"], color="lightgreen") \
                      .highlight_min(subset=["mean_reward"], color="#ffcccc") \
                      .format({"lr": "{:.2e}", "mean_reward": "{:.2f}", "std_reward": "{:.2f}"})

## 7. Bar Chart (Reward Comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor("#F5F0E8")
ax.set_facecolor("#F5F0E8")

labels  = [r["experiment"] for r in results]
means   = [r["mean_reward"] for r in results]
stds    = [r["std_reward"]  for r in results]
policies = [r["policy"] for r in results]

cnn_color = "#2ECC71"
mlp_color = "#E74C3C"
colors = [cnn_color if p == "CnnPolicy" else mlp_color for p in policies]

bars = ax.barh(labels, means, xerr=stds,
               color=colors, edgecolor="white", linewidth=0.8,
               capsize=4, error_kw={"ecolor": "#333", "linewidth": 1.2})

# Labels on bars
for bar, mean in zip(bars, means):
    x = bar.get_width()
    offset = 0.5 if x >= 0 else -0.5
    ax.text(x + offset, bar.get_y() + bar.get_height() / 2,
            f"{mean:.1f}", va="center", ha="left" if x >= 0 else "right",
            fontsize=9, fontweight="bold", color="#1a1a1a")

ax.axvline(0, color="#555", linewidth=1.2, linestyle="--", alpha=0.7)

ax.set_xlabel("Mean Reward (+/- std)", fontsize=12, labelpad=8)
ax.set_title("DQN Experiments — ALE/Boxing-v5 | Member: Nice",
             fontsize=14, fontweight="bold", pad=14)

cnn_patch = mpatches.Patch(color=cnn_color, label="CnnPolicy")
mlp_patch = mpatches.Patch(color=mlp_color, label="MlpPolicy")
ax.legend(handles=[cnn_patch, mlp_patch], loc="lower right", fontsize=10)

ax.invert_yaxis()
ax.grid(axis="x", linestyle="--", alpha=0.4, color="#aaa")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig("reward_comparison.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved → reward_comparison.png")

## 8. Episode Rewards Over Training (Training curves)

In [ ]:
def smooth(values, window=5):
    """Simple moving average."""
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")


fig, axes = plt.subplots(5, 2, figsize=(16, 20), sharex=False)
fig.patch.set_facecolor("#0F1923")
axes = axes.flatten()

palette = [
    "#00D2FF", "#FF6B6B", "#FFE66D", "#6BCB77", "#9B5DE5",
    "#F15BB5", "#FEE440", "#00BBF9", "#00F5D4", "#FF9A3C",
]

for idx, cfg in enumerate(EXPERIMENTS):
    ax = axes[idx]
    ax.set_facecolor("#1A2535")
    name = cfg["name"]
    records = all_curves.get(name, [])
    color = palette[idx]

    if records:
        ts  = [r["timestep"] for r in records]
        rws = [r["reward"]   for r in records]
        ax.plot(ts, rws, color=color, alpha=0.25, linewidth=0.8)
        sm = smooth(rws, window=max(1, len(rws) // 10))
        ts_sm = ts[:len(sm)]
        ax.plot(ts_sm, sm, color=color, linewidth=2.2, label="smoothed")
    else:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                ha="center", va="center", color="#aaa")

    mean_r = next(r["mean_reward"] for r in results if r["experiment"] == name)
    ax.set_title(f"{name}\nmean={mean_r:.1f}",
                 fontsize=9, fontweight="bold", color=color, pad=4)
    ax.tick_params(colors="#aaa", labelsize=7)
    ax.set_xlabel("Timestep", fontsize=7, color="#888")
    ax.set_ylabel("Episode Reward", fontsize=7, color="#888")
    for spine in ax.spines.values():
        spine.set_edgecolor("#2a3d55")
    ax.grid(linestyle="--", alpha=0.15, color="#aaa")

fig.suptitle("Training Curves — DQN ALE/Boxing-v5 | Member: Nice",
             fontsize=15, fontweight="bold", color="white", y=1.01)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print(" Saved, training_curves.png")

## 9. Results Summary

In [ ]:
print("\n" + "="*70)
print("NICE — EXPERIMENT SUMMARY")
print("="*70)

sorted_results = sorted(results, key=lambda x: x["mean_reward"], reverse=True)
for rank, r in enumerate(sorted_results, 1):
    marker = " best" if r["experiment"] == best_exp else ""
    print(f"{rank:2d}. {r['experiment']:35s} mean={r['mean_reward']:+6.2f} ± {r['std_reward']:.2f}{marker}")

print("\n" + "-"*70)
print("Key insights:")
print(f"  Best config  : {best_exp} (mean={best_mean:.2f})")
worst = min(results, key=lambda x: x["mean_reward"])
print(f"  Worst config : {worst['experiment']} (mean={worst['mean_reward']:.2f})")
print("  Zero gamma   : agent fully myopic — no future planning")
print("  Zero eps_end : fully greedy convergence — no residual exploration")
print("  Tiny LR      : near-zero updates — agent barely improves")
print("  MLP vs CNN   : MLP uses RAM obs, CNN uses pixel obs with frame-stacking")

print("\nArtifacts produced:")
for f in ["reward_comparison.png", "training_curves.png",
           "hyperparameter_table_nice.csv", "Nice_dqn_model.zip"]:
    exists = "yes" if Path(f).exists() else "no"
    print(f"  {exists} {f}")

---
## Appendix: Hyperparameter Table

| Experiment | Policy | lr | gamma | batch | eps_start | eps_end | eps_fraction | Note |
|---|---|---|---|---|---|---|---|---|
| exp01_baseline | CNN | 2.5e-4 | 0.95 | 64 | 1.0 | 0.02 | 0.20 | Moderate config — reference point |
| exp02_zero_eps_end | CNN | 2.5e-4 | 0.95 | 64 | 1.0 | **0.0** | 0.20 | Fully greedy at end; no residual exploration |
| exp03_zero_gamma | CNN | 2.5e-4 | **0.0** | 64 | 1.0 | 0.02 | 0.20 | Myopic agent; ignores future rewards |
| exp04_very_high_lr | CNN | **1e-3** | 0.95 | 64 | 1.0 | 0.02 | 0.20 | Aggressive updates; likely unstable |
| exp05_tiny_lr | CNN | **1e-6** | 0.95 | 64 | 1.0 | 0.02 | 0.20 | Near-zero LR; agent stagnates |
| exp06_large_batch_high_gamma | CNN | 2.5e-4 | **0.999** | **256** | 1.0 | 0.02 | 0.20 | Stable gradients + strong future valuation |
| exp07_instant_exploit | CNN | 2.5e-4 | 0.95 | 64 | 1.0 | 0.02 | **0.01** | Epsilon decays in first 1% — almost no exploration |
| exp08_full_explore | CNN | 2.5e-4 | 0.95 | 64 | 1.0 | 0.02 | **0.90** | Explores 90% of training; slow to exploit |
| exp09_best_guess | CNN | 2.5e-4 | **0.99** | 64 | 1.0 | **0.01** | 0.15 | Balanced config; expected best performance |
| exp10_mlp_ablation | **MLP** | 2.5e-4 | 0.99 | 64 | 1.0 | 0.01 | 0.15 | CNN vs MLP architecture comparison on RAM obs |